<a href="https://colab.research.google.com/github/candido100/Datos-masivos/blob/main/scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import requests
from bs4 import BeautifulSoup
import math
import sqlite3

    )
''')

# Función para insertar un inmueble en la base de datos
def insert_house(house_data):
    cursor.execute('''
        INSERT INTO inmuebles (imgUrl, price_house, title, address, size, rooms, url)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (house_data['imgUrl'], house_data['price_house'], house_data['title'],
          house_data['address'], house_data['size'], house_data['rooms'], house_data['url']))
    
    conn.commit()

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])  # Manejo de 'src' y 'data-src'
    price_house = house_html.find(class_="price-tag-fraction").text
    title = house_html.find(class_="ui-search-item__title").text
    address = house_html.find(class_="ui-search-item__group__element ui-search-item__location").text
    all_attributes = house_html.find_all("li", class_="ui-search-card-attributes__attribute")
    
    size = all_attributes[0].text if all_attributes else "N/A"
    rooms = all_attributes[1].text if len(all_attributes) > 1 else "N/A"
    
    url = house_html.find("a")["href"]
    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)  # Ajustado a 48 elementos por página
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")
    
    for house_html in houses:
        house_obj = obj_house(house_html)
        DAO2.insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)
C


In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble en la base de datos
def insert_house(house_data):
    print(f"Inserting house: {house_data['title']}, Precio: {house_data['price_house']}")
    # Aquí podrías insertar los datos en una base de datos.

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    address = house_html.find(class_="ui-search-item__group__element ui-search-item__location")
    address = address.text if address else "N/A"

    all_attributes = house_html.find_all("li", class_="ui-search-card-attributes__attribute")
    size = all_attributes[0].text if all_attributes else "N/A"
    rooms = all_attributes[1].text if len(all_attributes) > 1 else "N/A"

    url = house_html.find("a")["href"]
    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

Inserting house: Departamento En Venta En Lomas De Costa Azul Acapulco, Precio: $2,500,000
Inserting house: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante, Precio: $961,008
Inserting house: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo, Precio: $3,570,000
Inserting house: Departamento Residencial En Zona Diamante De Acapulco , Precio: $3,185,000
Inserting house: Pre Venta De Departamentos Prototipo Perla - Dream Diamante , Precio: $3,185,000
Inserting house: Departamentos En Pre Venta Con Amenidades Acuáticas Y Acceso, Precio: $1,834,854
Inserting house: Departamento De Lujo En Acapulco Con Club De Playa, Precio: $3,535,000
Inserting house: Departamento En Venta En Acapulco, Dream Diamante, Playa Bonfil, Club De Playa, Gimnasio, Canchas De Tenis, Paddle Y De Usos Múltiples, Precio: $3,570,000
Inserting house: Departamento En Venta, Con Exclusivo Club De Playa, En Acapulco, Precio: $1,834,800
Inser

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble en la base de datos
def insert_house(house_data):
    print(f"Inserting house: {house_data['title']}, Precio: {house_data['price_house']}, Superficie: {house_data['size']}")
    # Aquí podrías insertar los datos en una base de datos.

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    address = house_html.find(class_="ui-search-item__group__element ui-search-item__location")
    address = address.text if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    url = house_html.find("a")["href"]
    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)


Inserting house: Departamento En Venta En Lomas De Costa Azul Acapulco, Precio: $2,500,000, Superficie: 100 m² construidos
Inserting house: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante, Precio: $961,008, Superficie: 50 m² construidos
Inserting house: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo, Precio: $3,570,000, Superficie: 105 m² construidos
Inserting house: Departamento Residencial En Zona Diamante De Acapulco , Precio: $3,185,000, Superficie: 94 m² construidos
Inserting house: Pre Venta De Departamentos Prototipo Perla - Dream Diamante , Precio: $3,185,000, Superficie: 95 m² construidos
Inserting house: Departamentos En Pre Venta Con Amenidades Acuáticas Y Acceso, Precio: $1,834,854, Superficie: 76 m² construidos
Inserting house: Departamento De Lujo En Acapulco Con Club De Playa, Precio: $3,535,000, Superficie: 93 m² construidos
Inserting house: Departamento En Venta En Acapulco, Dream D

In [ ]:
import requests
from bs4 import BeautifulSoup
import math
import pandas as pd

# Lista para almacenar los resultados
houses_data = []

#---------------------------------------------------------------
# Función para insertar un inmueble en la base de datos (o en la lista)
def insert_house(house_data):
    houses_data.append(house_data)

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    address = house_html.find(class_="ui-search-item__group__element ui-search-item__location")
    address = address.text if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    url = house_html.find("a")["href"]
    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

#---------------------------------------------------------------
# Exportar los datos a un archivo de Excel
df = pd.DataFrame(houses_data)
df.to_excel("C:\\Users\\bandy\\Documents\\FONATUR\\inmuebles_acapulco.xlsx", index=False)

print("Datos exportados a inmuebles_acapulco.xlsx")

Datos exportados a inmuebles_acapulco.xlsx


In [ ]:
import requests
from bs4 import BeautifulSoup
import math
import pandas as pd

# Lista para almacenar los resultados
houses_data = []

#---------------------------------------------------------------
# Función para insertar un inmueble en la base de datos (o en la lista)
def insert_house(house_data):
    houses_data.append(house_data)

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    address = house_html.find(class_="ui-search-item__group__element ui-search-item__location")
    address = address.text if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    url = house_html.find("a")["href"]
    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

#---------------------------------------------------------------
# Exportar los datos a un archivo CSV
df = pd.DataFrame(houses_data)
df.to_csv(r"C:\Users\bandy\Documents\FONATUR\inmuebles_acapulco.csv", index=False)

print("Datos exportados a C:\\Users\\bandy\\Documents\\FONATUR\\inmuebles_acapulco.csv")


Datos exportados a C:\Users\bandy\Documents\FONATUR\inmuebles_acapulco.csv


In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble en la base de datos
def insert_house(house_data):
    print(f"Inserting house: {house_data['title']}, Precio: {house_data['price_house']}, Superficie: {house_data['size']}, Ubicación: {house_data['address']}")
    # Aquí podrías insertar los datos en una base de datos.

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    url = house_html.find("a")["href"]
    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

Inserting house: Departamento En Venta En Lomas De Costa Azul Acapulco, Precio: $2,500,000, Superficie: 100 m² construidos, Ubicación: Lomas De Costa Azul, Lomas De Cost...
Inserting house: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante, Precio: $961,008, Superficie: 50 m² construidos, Ubicación: Tampico Carretera Barra Vieja, Pla...
Inserting house: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo, Precio: $3,570,000, Superficie: 105 m² construidos, Ubicación: Alfredo V Bonfil, Aeropuerto, 3989...
Inserting house: Departamento Residencial En Zona Diamante De Acapulco , Precio: $3,185,000, Superficie: 94 m² construidos, Ubicación: C. Revolución 272, Bonfil, Acapulc...
Inserting house: Pre Venta De Departamentos Prototipo Perla - Dream Diamante , Precio: $3,185,000, Superficie: 95 m² construidos, Ubicación: Boulevard Barra Vieja Km, Alfredo ...
Inserting house: Departamentos En Pre Venta Con Amenidade

In [ ]:
import requests
from bs4 import BeautifulSoup
import math
import csv
import os

#---------------------------------------------------------------
# Ruta donde se guardará el archivo CSV
output_path = r'C:\Users\bandy\Documents\FONATUR\inmuebles_acapulco.csv'

#---------------------------------------------------------------
# Función para insertar un inmueble en un archivo CSV
def insert_house(house_data, csv_writer):
    csv_writer.writerow([house_data['title'], house_data['price_house'], house_data['size'], house_data['rooms'], house_data['address'], house_data['url']])

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    url = house_html.find("a")["href"]
    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls, csv_writer):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj, csv_writer)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Crear el archivo CSV y escribir el encabezado
with open(output_path, mode='w', newline='', encoding='utf-8') as file:
    csv_writer = csv.writer(file)

    # Escribir la cabecera del CSV
    csv_writer.writerow(["Title", "Price", "Size", "Rooms", "Address", "URL"])

    # Iterar sobre todas las páginas y scrapear
    for i in range(1, page_amount + 1):
        url_page = page(i)
        scrap_url(url_page, csv_writer)

print(f"Archivo CSV generado correctamente en {output_path}")
df.to_csv(r"C:\Users\bandy\Documents\FONATUR\inmuebles_acapulco.csv", index=False)


Archivo CSV generado correctamente en C:\Users\bandy\Documents\FONATUR\inmuebles_acapulco.csv


In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print("="*50)

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    url = house_html.find("a")["href"]

    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

print("Datos extraídos correctamente.")

Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D4b607cbe-cec1-4a36-9d70-4c9e119e5c07
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carretera Barra Vieja, Pla...
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D4b607cbe-cec1-4a36-9d70-4c9e119e5c07
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: $3,570,000
Size: 105 m² construidos
Rooms: 4 recámara

In [ ]:
import requests
from bs4 import BeautifulSoup
import math
import re

#---------------------------------------------------------------
# Función para insertar un inmueble en una base de datos
def insert_house(house_data):
    print(f"Inserting house: {house_data['title']}, Precio: {house_data['price_house']}, Superficie: {house_data['size']}, Coordenadas: {house_data['coordinates']}")
    # Aquí podrías insertar los datos en una base de datos.

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    address = house_html.find(class_="ui-search-item__group__element ui-search-item__location")
    address = address.text if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    url = house_html.find("a")["href"]

    # ---------------------------------------------------------------
    # Nueva sección para obtener las coordenadas del mapa
    r_details = requests.get(url)
    detail_soup = BeautifulSoup(r_details.text, 'lxml')

    # Buscar las coordenadas dentro del iframe o los scripts que cargan el mapa
    # Esta es una búsqueda tentativa, ya que las coordenadas pueden estar en diferentes lugares
    coordinates = "N/A"
    map_iframe = detail_soup.find("iframe", src=re.compile("google.com/maps"))
    if map_iframe:
        map_src = map_iframe['src']
        # Intentar encontrar las coordenadas en la URL del mapa
        match = re.search(r'@(-?\d+\.\d+),(-?\d+\.\d+)', map_src)
        if match:
            coordinates = f"{match.group(1)}, {match.group(2)}"

    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url, "coordinates": coordinates}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

Inserting house: Departamento En Venta En Lomas De Costa Azul Acapulco, Precio: $2,500,000, Superficie: 100 m² construidos, Coordenadas: N/A
Inserting house: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante, Precio: $961,008, Superficie: 50 m² construidos, Coordenadas: N/A
Inserting house: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo, Precio: $3,570,000, Superficie: 105 m² construidos, Coordenadas: N/A
Inserting house: Departamento Residencial En Zona Diamante De Acapulco , Precio: $3,185,000, Superficie: 94 m² construidos, Coordenadas: N/A
Inserting house: Pre Venta De Departamentos Prototipo Perla - Dream Diamante , Precio: $3,185,000, Superficie: 95 m² construidos, Coordenadas: N/A
Inserting house: Departamentos En Pre Venta Con Amenidades Acuáticas Y Acceso, Precio: $1,834,854, Superficie: 76 m² construidos, Coordenadas: N/A
Inserting house: Departamento De Lujo En Acapulco Con Club De Playa, P

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print(f"Google Maps URL: {house_data['map_url']}")
    print("="*50)

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    url = house_html.find("a")["href"]

    # ---------------------------------------------------------------
    # Extraer la URL del mapa de Google desde la página de detalles del inmueble
    r_details = requests.get(url)
    detail_soup = BeautifulSoup(r_details.text, 'lxml')

    map_url = "N/A"
    map_link = detail_soup.find("a", string="Abrir esta área en Google Maps")
    if map_link:
        map_url = map_link['href']

    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url, "map_url": map_url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

print("Datos extraídos correctamente.")


Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D97d265e1-9dfc-40fb-b8af-b16567a8f68f
Google Maps URL: N/A
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carretera Barra Vieja, Pla...
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D97d265e1-9dfc-40fb-b8af-b16567a8f68f
Google Maps URL: N/A
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: $3,570,000


KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print(f"Google Maps URL: {house_data['map_url']}")
    print("="*50)

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    url = house_html.find("a")["href"]

    # ---------------------------------------------------------------
    # Extraer la URL del mapa de Google desde la página de detalles del inmueble
    r_details = requests.get(url)
    detail_soup = BeautifulSoup(r_details.text, 'lxml')

    map_url = "N/A"

    # Buscar el contenedor que incluye la URL del mapa
    map_link = detail_soup.find("a", {"title": "Abrir esta área en Google Maps"})

    if map_link:
        map_url = map_link['href']

    return {"imgUrl": imgUrl, "price_house": price_house, "title": title, "address": address, "size": size, "rooms": rooms, "url": url, "map_url": map_url}

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

print("Datos extraídos correctamente.")

Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D8c616a32-1dda-4320-abed-2d38c9b7a463
Google Maps URL: N/A
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carretera Barra Vieja, Pla...
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D8c616a32-1dda-4320-abed-2d38c9b7a463
Google Maps URL: N/A
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: $3,570,000


KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print(f"URL: {house_data['url']}")
    print("="*50)

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    url = house_html.find("a")["href"]

    # Extraer la URL de Google Maps
    google_maps_link = house_html.find("a", {"aria-label": "Abrir esta área en Google Maps"})
    google_maps_url = google_maps_link["href"] if google_maps_link else "N/A"

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": url
    }

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

print("Datos extraídos correctamente.")

Se truncaron las últimas líneas 5000 del resultado de transmisión.
Price: $3,690,000
Size: 110 m² construidos
Rooms: 3 recámaras
Address: Costera De Las Palmas, Playa Diama...
Google Maps URL: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3113424926-departamento-en-venta-en-kabah-acapulco-playa-diamante-_JM#position%3D48%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3Dd521606f-450f-4d39-a7b6-404f6ab68b20
Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
Google Maps URL: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3Df477037f-70eb-4610-8080-3637c7304efa
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carret

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print(f"URL: {house_data['url']}")
    print("="*50)

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    url = house_html.find("a")["href"]

    # Extraer la URL de Google Maps (buscando todas las etiquetas <a> que contengan "maps.google.com")
    google_maps_url = "N/A"
    for link in house_html.find_all('a', href=True):
        if "maps.google.com" in link['href']:
            google_maps_url = link['href']
            break

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": url
    }

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

print("Datos extraídos correctamente.")

Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
Google Maps URL: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D9155b5ee-cad7-470a-97f0-3004bc1195d9
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carretera Barra Vieja, Pla...
Google Maps URL: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D9155b5ee-cad7-470a-97f0-3004bc1195d9
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: $3,570,000


KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print(f"URL: {house_data['url']}")
    print("="*50)

#---------------------------------------------------------------
# Función para extraer la URL de Google Maps desde la página del inmueble
def get_google_maps_url(inmueble_url):
    r = requests.get(inmueble_url)
    inmueble_soup = BeautifulSoup(r.text, 'lxml')

    google_maps_url = "N/A"

    # Buscar enlaces que contengan "maps.google.com"
    for link in inmueble_soup.find_all('a', href=True):
        if "maps.google.com" in link['href']:
            google_maps_url = link['href']
            break

    return google_maps_url

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    inmueble_url = house_html.find("a")["href"]

    # Extraer la URL de Google Maps desde la página del inmueble
    google_maps_url = get_google_maps_url(inmueble_url)

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": inmueble_url
    }

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

print("Datos extraídos correctamente.")

Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
Google Maps URL: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D01de9313-b7dd-4e92-89aa-bdd83d8e879a
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carretera Barra Vieja, Pla...
Google Maps URL: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D01de9313-b7dd-4e92-89aa-bdd83d8e879a
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: $3,570,000


KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print("="*50)

#---------------------------------------------------------------
# Función para extraer todas las etiquetas <a> y sus href
def get_google_maps_url(inmueble_url):
    r = requests.get(inmueble_url)
    inmueble_soup = BeautifulSoup(r.text, 'lxml')

    google_maps_url = "N/A"

    # Buscar todas las etiquetas <a> que tengan un atributo href
    for link in inmueble_soup.find_all('a', href=True):
        # Verificamos si el enlace tiene "maps.google.com"
        if "maps.google.com" in link['href']:
            google_maps_url = link['href']
            break

    return google_maps_url

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    inmueble_url = house_html.find("a")["href"]

    # Extraer todas las URLs de <a> de la página del inmueble
    google_maps_url = get_google_maps_url(inmueble_url)

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": inmueble_url
    }

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos 'li'
houses = soup.find_all("li", class_="ui-search-layout__item")

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Obtener el número de elementos y páginas
houses_amount = float(soup.find(class_="ui-search-search-result__quantity-results").text.split(" ")[0].replace(".", ""))
page_amount = math.ceil(houses_amount / 48)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    scrap_url(url_page)

print("Datos extraídos correctamente.")

AttributeError: 'NoneType' object has no attribute 'text'

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print("="*50)

#---------------------------------------------------------------
# Función para extraer todas las etiquetas <a> y sus href
def get_google_maps_url(inmueble_url):
    r = requests.get(inmueble_url)
    inmueble_soup = BeautifulSoup(r.text, 'lxml')

    google_maps_url = "N/A"

    # Buscar todas las etiquetas <a> que tengan un atributo href
    for link in inmueble_soup.find_all('a', href=True):
        # Verificamos si el enlace tiene "maps.google.com"
        if "maps.google.com" in link['href']:
            google_maps_url = link['href']
            break

    return google_maps_url

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    inmueble_url = house_html.find("a")["href"]

    # Extraer todas las URLs de <a> de la página del inmueble
    google_maps_url = get_google_maps_url(inmueble_url)

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": inmueble_url
    }

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Verificar si encontramos el total de resultados
quantity_result = soup.find(class_="ui-search-search-result__quantity-results")

if quantity_result:
    houses_amount = float(quantity_result.text.split(" ")[0].replace(".", ""))
    page_amount = math.ceil(houses_amount / 48)
else:
    print("No se pudo encontrar la cantidad de resultados.")
    houses_amount = 0
    page_amount = 1

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')
    houses = soup.find_all("li", class_="ui-search-layout__item")

    if not houses:
        print("No se encontraron inmuebles en esta página.")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    print(f"Scraping página {i}...")
    scrap_url(url_page)

print("Datos extraídos correctamente.")

No se pudo encontrar la cantidad de resultados.
Scraping página 1...
No se encontraron inmuebles en esta página.
Datos extraídos correctamente.


In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print("="*50)

#---------------------------------------------------------------
# Función para extraer todas las etiquetas <a> y sus href
def get_google_maps_url(inmueble_url):
    r = requests.get(inmueble_url)
    inmueble_soup = BeautifulSoup(r.text, 'lxml')

    google_maps_url = "N/A"

    # Buscar todas las etiquetas <a> que tengan un atributo href
    for link in inmueble_soup.find_all('a', href=True):
        # Verificamos si el enlace tiene "maps.google.com"
        if "maps.google.com" in link['href']:
            google_maps_url = link['href']
            break

    return google_maps_url

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    inmueble_url = house_html.find("a")["href"]

    # Extraer todas las URLs de <a> de la página del inmueble
    google_maps_url = get_google_maps_url(inmueble_url)

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": inmueble_url
    }

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Imprimir una parte del HTML para inspección
print(soup.prettify()[:1000])  # Imprime los primeros 1000 caracteres del HTML

#---------------------------------------------------------------
# Verificar si encontramos el total de resultados
quantity_result = soup.find(class_="ui-search-search-result__quantity-results")

if quantity_result:
    houses_amount = float(quantity_result.text.split(" ")[0].replace(".", ""))
    page_amount = math.ceil(houses_amount / 48)
else:
    print("No se pudo encontrar la cantidad de resultados.")
    houses_amount = 0
    page_amount = 1

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')

    # Imprimir los primeros 1000 caracteres del HTML de la página
    print(soup.prettify()[:1000])  # Para verificar si está cargando bien

    houses = soup.find_all("li", class_="ui-search-layout__item")

    if not houses:
        print("No se encontraron inmuebles en esta página.")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    print(f"Scraping página {i}...")
    scrap_url(url_page)

print("Datos extraídos correctamente.")

<!DOCTYPE html>
<html lang="es-MX">
 <head>
  <link href="https://www.google-analytics.com" rel="preconnect"/>
  <link href="https://www.google.com" rel="preconnect"/>
  <link href="https://data.mercadolibre.com" rel="preconnect"/>
  <link href="https://http2.mlstatic.com" rel="preconnect"/>
  <link href="https://stats.g.doubleclick.net" rel="preconnect"/>
  <link href="https://analytics.mercadolibre.com.mx" rel="preconnect"/>
  <link href="https://analytics.mercadolibre.com" rel="preconnect"/>
  <link href="https://www.google.com.mx" rel="preconnect"/>
  <script nonce="G3QzSNbwJeFG7iHtqAXWSg==" type="text/javascript">
   window.NREUM||(NREUM={});NREUM.info = {"agent":"","beacon":"bam.nr-data.net","errorBeacon":"bam.nr-data.net","licenseKey":"NRBR-8547a290c864571ffcc","applicationID":"1729522169","agentToken":null,"applicationTime":411.209757,"transactionName":"bgQDMEcFXkJZBkYNWldOJBxFFlVCSw9BS3J8NU5LHw==","queueTime":0,"ttGuid":"1391a9ed3d7bf81e"}; (window.NREUM||(NREUM={})).init={pri

KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import math

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print("="*50)

#---------------------------------------------------------------
# Función para extraer el enlace de Google Maps desde la página del inmueble
def get_google_maps_url(inmueble_url):
    r = requests.get(inmueble_url)
    inmueble_soup = BeautifulSoup(r.text, 'lxml')

    google_maps_url = "N/A"

    # Buscar todas las etiquetas <a> que tengan un atributo href y que contengan 'maps.google'
    map_link = inmueble_soup.find('a', href=lambda href: href and "maps.google.com" in href)

    if map_link:
        google_maps_url = map_link['href']

    return google_maps_url

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"])

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    inmueble_url = house_html.find("a")["href"]

    # Extraer todas las URLs de <a> de la página del inmueble
    google_maps_url = get_google_maps_url(inmueble_url)

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": inmueble_url
    }

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Verificar si encontramos el total de resultados
quantity_result = soup.find(class_="ui-search-search-result__quantity-results")

if quantity_result:
    houses_amount = float(quantity_result.text.split(" ")[0].replace(".", ""))
    page_amount = math.ceil(houses_amount / 48)
else:
    print("No se pudo encontrar la cantidad de resultados.")
    houses_amount = 0
    page_amount = 1

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    r = requests.get(urls)
    data = r.text
    soup = BeautifulSoup(data, 'lxml')

    houses = soup.find_all("li", class_="ui-search-layout__item")

    if not houses:
        print("No se encontraron inmuebles en esta página.")

    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    print(f"Scraping página {i}...")
    scrap_url(url_page)

print("Datos extraídos correctamente.")

Scraping página 1...
Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3De248be1e-e473-422d-8431-ee245d279918
Google Maps URL: N/A
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carretera Barra Vieja, Pla...
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3De248be1e-e473-422d-8431-ee245d279918
Google Maps URL: N/A
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deporti

KeyboardInterrupt: 

In [ ]:
import requests
from bs4 import BeautifulSoup
import math
import time

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Address: {house_data['address']}")
    print(f"URL: {house_data['url']}")
    print(f"Google Maps URL: {house_data['google_maps_url']}")
    print("="*50)

#---------------------------------------------------------------
# Función para extraer el enlace de Google Maps desde la página del inmueble
def get_google_maps_url(inmueble_url):
    try:
        r = requests.get(inmueble_url)
        inmueble_soup = BeautifulSoup(r.text, 'lxml')

        google_maps_url = "N/A"

        # Buscar todas las etiquetas <a> que tengan un atributo href y que contengan 'maps.google'
        map_link = inmueble_soup.find('a', href=lambda href: href and "maps.google.com" in href)

        if map_link:
            google_maps_url = map_link['href']

        return google_maps_url
    except Exception as e:
        print(f"Error al obtener la URL de Google Maps: {e}")
        return "N/A"

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    imgUrl = house_html.find("img").get("data-src", house_html.find("img")["src"]) if house_html.find("img") else "N/A"

    # Buscar el contenedor del precio usando la clase correcta
    price_house = house_html.find(class_="ui-search-item__group--price")
    price_house = price_house.text.strip() if price_house else "N/A"

    title = house_html.find(class_="ui-search-item__title")
    title = title.text if title else "N/A"

    # Se ha añadido la extracción de la ubicación (dirección)
    address = house_html.find("div", class_="ui-search-item__location-container-grid")
    address = address.text.strip() if address else "N/A"

    # Buscar el contenedor de atributos, donde está la superficie
    attributes_container = house_html.find("div", class_="ui-search-item__attributes-container-grid")
    size = "N/A"
    if attributes_container:
        attributes = attributes_container.find_all("li")
        for attribute in attributes:
            if "m²" in attribute.text:
                size = attribute.text.strip()
                break

    rooms = "N/A"
    for attribute in attributes:
        if "recámara" in attribute.text:
            rooms = attribute.text.strip()
            break

    # Extraer la URL del inmueble
    inmueble_url = house_html.find("a")["href"]

    # Extraer todas las URLs de <a> de la página del inmueble
    google_maps_url = get_google_maps_url(inmueble_url)

    return {
        "imgUrl": imgUrl,
        "price_house": price_house,
        "title": title,
        "address": address,
        "size": size,
        "rooms": rooms,
        "google_maps_url": google_maps_url,
        "url": inmueble_url
    }

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Verificar si encontramos el total de resultados
quantity_result = soup.find(class_="ui-search-search-result__quantity-results")

if quantity_result:
    houses_amount = float(quantity_result.text.split(" ")[0].replace(".", ""))
    page_amount = math.ceil(houses_amount / 48)
else:
    print("No se pudo encontrar la cantidad de resultados.")
    houses_amount = 0
    page_amount = 1

#---------------------------------------------------------------
# Función para generar la URL de cada página
def page(pageNumber):
    initial_range = 1 + 48 * (pageNumber - 1)
    page_url = f"https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/_Desde_{initial_range}"
    return page_url

#---------------------------------------------------------------
# Función para extraer inmuebles de una página
def scrap_url(urls):
    try:
        r = requests.get(urls)
        data = r.text
        soup = BeautifulSoup(data, 'lxml')

        houses = soup.find_all("li", class_="ui-search-layout__item")

        if not houses:
            print("No se encontraron inmuebles en esta página.")

        for house_html in houses:
            house_obj = obj_house(house_html)
            insert_house(house_obj)
    except Exception as e:
        print(f"Error al acceder a la página, código de estado: {r.status_code}")
        print(f"Detalles del error: {e}")

#---------------------------------------------------------------
# Iterar sobre todas las páginas y scrapear
for i in range(1, page_amount + 1):
    url_page = page(i)
    print(f"Scraping página {i}...")
    scrap_url(url_page)

    # Esperar 2 segundos entre solicitudes
    time.sleep(2)

print("Datos extraídos correctamente.")


Scraping página 1...
Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: $2,500,000
Size: 100 m² construidos
Rooms: 2 recámaras
Address: Lomas De Costa Azul, Lomas De Cost...
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D8b89409e-59a7-452c-aecc-cf79580032f0
Google Maps URL: N/A
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: $961,008
Size: 50 m² construidos
Rooms: 2 recámaras
Address: Tampico Carretera Barra Vieja, Pla...
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D8b89409e-59a7-452c-aecc-cf79580032f0
Google Maps URL: N/A
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deporti

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Lista de URLs de las propiedades que deseas procesar
property_urls = [
    "https://departamento.mercadolibre.com.mx/MLM-3113424926-departamento-en-venta-en-kabah-acapulco-playa-diamante_JM",
    # Agrega las demás URLs aquí...
]

# Lista para almacenar los resultados
data = []

# Función para extraer el link de Google Maps de una propiedad
def extract_google_maps_url(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Encuentra todos los enlaces
    links = soup.find_all('a', href=True)

    # Busca el link de Google Maps
    for link in links:
        href = link['href']
        if 'google.com/maps' in href:
            return href
    return None

# Iterar sobre las URLs de las propiedades
for property_url in property_urls:
    google_maps_url = extract_google_maps_url(property_url)
    data.append({'Property URL': property_url, 'Google Maps URL': google_maps_url})

# Crear un DataFrame
df = pd.DataFrame(data)

# Guardar los resultados en un archivo CSV
df.to_csv('property_google_maps_links.csv', index=False)

print("Extracción completada y archivo CSV generado.")

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# Configura Selenium para abrir el navegador (Chrome en este caso)
driver_path = '/path/to/chromedriver'  # Cambia esta ruta al path de tu driver de Selenium
driver = webdriver.Chrome(executable_path=driver_path)

# URL de MercadoLibre que queremos scrapear
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
driver.get(url)

# Esperamos a que se cargue la página
time.sleep(5)

# Extraer todos los links de los inmuebles
inmuebles = driver.find_elements(By.CSS_SELECTOR, "li.ui-search-layout__item a.ui-search-link")

# Recorremos cada inmueble para extraer la URL de Google Maps
for inmueble in inmuebles:
    inmueble_url = inmueble.get_attribute('href')
    print(f"Inmueble URL: {inmueble_url}")

    # Ir a la página del inmueble
    driver.get(inmueble_url)

    # Esperamos que la página cargue completamente
    time.sleep(3)

    # Intentar obtener el link de Google Maps (si está disponible)
    try:
        google_maps_link = driver.find_element(By.XPATH, "//a[contains(@href, 'maps.google.com')]").get_attribute('href')
        print(f"Google Maps URL: {google_maps_link}")
    except:
        print("Google Maps URL: N/A")

    print("="*50)

    # Volver a la página principal
    driver.back()
    time.sleep(2)

# Cerramos el navegador al finalizar
driver.quit()

ModuleNotFoundError: No module named 'selenium'

In [ ]:
pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.0 MB/s eta 0:00:00


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# Configura Selenium para abrir el navegador (Chrome en este caso)
driver_path = '/path/to/chromedriver'  # Cambia esta ruta al path de tu driver de Selenium
driver = webdriver.Chrome(executable_path=driver_path)

# URL de MercadoLibre que queremos scrapear
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
driver.get(url)

# Esperamos a que se cargue la página
time.sleep(5)

# Extraer todos los links de los inmuebles
inmuebles = driver.find_elements(By.CSS_SELECTOR, "li.ui-search-layout__item a.ui-search-link")

# Recorremos cada inmueble para extraer la URL de Google Maps
for inmueble in inmuebles:
    inmueble_url = inmueble.get_attribute('href')
    print(f"Inmueble URL: {inmueble_url}")

    # Ir a la página del inmueble
    driver.get(inmueble_url)

    # Esperamos que la página cargue completamente
    time.sleep(3)

    # Intentar obtener el link de Google Maps (si está disponible)
    try:
        google_maps_link = driver.find_element(By.XPATH, "//a[contains(@href, 'maps.google.com')]").get_attribute('href')
        print(f"Google Maps URL: {google_maps_link}")
    except:
        print("Google Maps URL: N/A")

    print("="*50)

    # Volver a la página principal
    driver.back()
    time.sleep(2)

# Cerramos el navegador al finalizar
driver.quit()

TypeError: WebDriver.__init__() got an unexpected keyword argument 'executable_path'

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time

# Configura Selenium para abrir el navegador (Chrome en este caso)
driver_path = '/path/to/chromedriver'  # Cambia esta ruta al path de tu ChromeDriver
service = Service(driver_path)
driver = webdriver.Chrome(service=service)

# URL de MercadoLibre que queremos scrapear
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
driver.get(url)

# Esperamos a que se cargue la página
time.sleep(5)

# Extraer todos los links de los inmuebles
inmuebles = driver.find_elements(By.CSS_SELECTOR, "li.ui-search-layout__item a.ui-search-link")

# Recorremos cada inmueble para extraer la URL de Google Maps
for inmueble in inmuebles:
    inmueble_url = inmueble.get_attribute('href')
    print(f"Inmueble URL: {inmueble_url}")

    # Ir a la página del inmueble
    driver.get(inmueble_url)

    # Esperamos que la página cargue completamente
    time.sleep(3)

    # Intentar obtener el link de Google Maps (si está disponible)
    try:
        google_maps_link = driver.find_element(By.XPATH, "//a[contains(@href, 'maps.google.com')]").get_attribute('href')
        print(f"Google Maps URL: {google_maps_link}")
    except:
        print("Google Maps URL: N/A")

    print("="*50)

    # Volver a la página principal
    driver.back()
    time.sleep(2)

# Cerramos el navegador al finalizar
driver.quit()

NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


In [ ]:
# Instalar las dependencias necesarias
!apt-get update
!apt install -y chromium-chromedriver
!pip install selenium

# Configuración para usar Chrome en Colab
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# Configurar opciones de Chrome para Colab
chrome_options = Options()
chrome_options.add_argument("--headless")  # Ejecutar Chrome en modo sin interfaz gráfica
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Iniciar el servicio de Chrome
service = Service('/usr/lib/chromium-browser/chromedriver')
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL de MercadoLibre que queremos scrapear
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
driver.get(url)

# Esperamos a que se cargue la página
time.sleep(5)

# Extraer todos los links de los inmuebles
inmuebles = driver.find_elements(By.CSS_SELECTOR, "li.ui-search-layout__item a.ui-search-link")

# Recorremos cada inmueble para extraer la URL de Google Maps
for inmueble in inmuebles:
    inmueble_url = inmueble.get_attribute('href')
    print(f"Inmueble URL: {inmueble_url}")

    # Ir a la página del inmueble
    driver.get(inmueble_url)

    # Esperamos que la página cargue completamente
    time.sleep(3)

    # Intentar obtener el link de Google Maps (si está disponible)
    try:
        google_maps_link = driver.find_element(By.XPATH, "//a[contains(@href, 'maps.google.com')]").get_attribute('href')
        print(f"Google Maps URL: {google_maps_link}")
    except:
        print("Google Maps URL: N/A")

    print("="*50)

    # Volver a la página principal
    driver.back()
    time.sleep(2)

# Cerramos el navegador al finalizar
driver.quit()

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [966 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Ign:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:14 https

WebDriverException: Message: Service /usr/lib/chromium-browser/chromedriver unexpectedly exited. Status code was: 1


In [ ]:
# Descargar e instalar Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -f install

# Descargar la versión correcta de ChromeDriver
!wget https://chromedriver.storage.googleapis.com/114.0.5735.90/chromedriver_linux64.zip
!unzip chromedriver_linux64.zip
!chmod +x chromedriver
!mv chromedriver /usr/local/bin/chromedriver

# Instalar Selenium
!pip install selenium
Después de ejecutar este bloque de código para la instalación, puedes usar el siguiente código Python para el scraping:

python
Copiar código
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# Configurar las opciones de Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")  # Ejecutar en modo sin interfaz gráfica
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument('--disable-gpu')

# Definir el path del ChromeDriver
service = Service('/usr/local/bin/chromedriver')
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL de MercadoLibre que queremos scrapear
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
driver.get(url)

# Esperamos a que se cargue la página
time.sleep(5)

# Extraer todos los links de los inmuebles
inmuebles = driver.find_elements(By.CSS_SELECTOR, "li.ui-search-layout__item a.ui-search-link")

# Recorremos cada inmueble para extraer la URL de Google Maps
for inmueble in inmuebles:
    inmueble_url = inmueble.get_attribute('href')
    print(f"Inmueble URL: {inmueble_url}")

    # Ir a la página del inmueble
    driver.get(inmueble_url)

    # Esperamos que la página cargue completamente
    time.sleep(3)

    # Intentar obtener el link de Google Maps (si está disponible)
    try:
        google_maps_link = driver.find_element(By.XPATH, "//a[contains(@href, 'maps.google.com')]").get_attribute('href')
        print(f"Google Maps URL: {google_maps_link}")
    except:
        print("Google Maps URL: N/A")

    print("="*50)

    # Volver a la página principal
    driver.back()
    time.sleep(2)

# Cerramos el navegador al finalizar
driver.quit()

SyntaxError: invalid syntax (<ipython-input-44-a6fba0943dae>, line 14)

In [ ]:
# Descargar e instalar Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -f install

# Descargar la versión correcta de ChromeDriver
!wget https://chromedriver.storage.googleapis.com/114.0.5735.90/chromedriver_linux64.zip
!unzip chromedriver_linux64.zip
!chmod +x chromedriver
!mv chromedriver /usr/local/bin/chromedriver

# Instalar Selenium
!pip install selenium


--2024-09-06 05:52:14--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 64.233.187.91, 64.233.187.136, 64.233.187.190, ...
Connecting to dl.google.com (dl.google.com)|64.233.187.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 110842868 (106M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 105.71M   156MB/s    in 0.7s    

2024-09-06 05:52:15 (156 MB/s) - ‘google-chrome-stable_current_amd64.deb’ saved [110842868/110842868]

Selecting previously unselected package google-chrome-stable.
(Reading database ... 124063 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (128.0.6613.119-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libvulkan1; however:
  Package l

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# Configurar opciones de Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")  # Ejecutar en modo sin interfaz gráfica
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument('--disable-gpu')

# Definir el path del ChromeDriver
service = Service('/usr/local/bin/chromedriver')
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL de MercadoLibre que queremos scrapear
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"
driver.get(url)

# Esperamos a que se cargue la página
time.sleep(5)

# Extraer todos los links de los inmuebles
inmuebles = driver.find_elements(By.CSS_SELECTOR, "li.ui-search-layout__item a.ui-search-link")

# Recorremos cada inmueble para extraer la URL de Google Maps
for inmueble in inmuebles:
    inmueble_url = inmueble.get_attribute('href')
    print(f"Inmueble URL: {inmueble_url}")

    # Ir a la página del inmueble
    driver.get(inmueble_url)

    # Esperamos que la página cargue completamente
    time.sleep(3)

    # Intentar obtener el link de Google Maps (si está disponible)
    try:
        google_maps_link = driver.find_element(By.XPATH, "//a[contains(@href, 'maps.google.com')]").get_attribute('href')
        print(f"Google Maps URL: {google_maps_link}")
    except:
        print("Google Maps URL: N/A")

    print("="*50)

    # Volver a la página principal
    driver.back()
    time.sleep(2)

# Cerramos el navegador al finalizar
driver.quit()


SessionNotCreatedException: Message: session not created: This version of ChromeDriver only supports Chrome version 114
Current browser version is 128.0.6613.119 with binary path /usr/bin/google-chrome
Stacktrace:
#0 0x5d19b35fe4e3 <unknown>
#1 0x5d19b332dc76 <unknown>
#2 0x5d19b335b04a <unknown>
#3 0x5d19b33564a1 <unknown>
#4 0x5d19b3353029 <unknown>
#5 0x5d19b3391ccc <unknown>
#6 0x5d19b339147f <unknown>
#7 0x5d19b3388de3 <unknown>
#8 0x5d19b335e2dd <unknown>
#9 0x5d19b335f34e <unknown>
#10 0x5d19b35be3e4 <unknown>
#11 0x5d19b35c23d7 <unknown>
#12 0x5d19b35ccb20 <unknown>
#13 0x5d19b35c3023 <unknown>
#14 0x5d19b35911aa <unknown>
#15 0x5d19b35e76b8 <unknown>
#16 0x5d19b35e7847 <unknown>
#17 0x5d19b35f7243 <unknown>
#18 0x7db9dec99ac3 <unknown>


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# Ruta donde está el chromedriver
driver_path = r'C:\Program Files\Google\Chrome\Application\chromedriver.exe'

# Configurar las opciones de Chrome (por ejemplo, para ejecutarlo en segundo plano)
chrome_options = Options()
chrome_options.add_argument("--headless")  # Ejecuta Chrome en modo headless, sin abrir ventana

# Iniciar el servicio de Chrome
service = Service(driver_path)

# Iniciar el driver de Chrome
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL que queremos scrapear
url = "https://www.example.com"
driver.get(url)

# Aquí iría el código para extraer los datos de la página
print(driver.page_source)  # Esto imprime el código HTML de la página

# Cerrar el navegador
driver.quit()


NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# Ruta donde está el chromedriver
driver_path = r'C:\\chromedriver\\chromedriver.exe'  # Ruta corregida

# Configurar las opciones de Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")  # Opcional: ejecutar en modo headless

# Iniciar el servicio de Chrome con la ruta correcta
service = Service(driver_path)

# Iniciar el driver de Chrome
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL que queremos scrapear
url = "https://www.example.com"
driver.get(url)

# Aquí iría el código para extraer los datos de la página
print(driver.page_source)  # Imprime el código HTML de la página

# Cerrar el navegador
driver.quit()


NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# Ruta donde está el chromedriver (Asegúrate que sea correcta)
driver_path = r'C:\Program Files\Google\Chrome\Application'

# Configurar las opciones de Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")  # Esto es opcional si quieres ejecutar en modo headless

# Iniciar el servicio de Chrome con la ruta correcta
service = Service(driver_path)

# Iniciar el driver de Chrome
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL que queremos scrapear
url = "https://www.example.com"
driver.get(url)

# Aquí iría el código para extraer los datos de la página
print(driver.page_source)  # Imprime el código HTML de la página

# Cerrar el navegador
driver.quit()


NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# Especificar la ruta correcta al archivo chromedriver.exe
driver_path = r'C:\Program Files\Google\Chrome\Application\chromedriver.exe'  # Cambia esto a la ruta correcta en tu sistema

# Configurar las opciones de Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")  # Opcional para ejecutar en modo headless (sin interfaz gráfica)

# Iniciar el servicio de ChromeDriver con la ruta correcta
service = Service(driver_path)

# Inicializar el controlador de Chrome
driver = webdriver.Chrome(service=service, options=chrome_options)

# Acceder a la página deseada
url = "https://www.example.com"
driver.get(url)

# Imprimir el código HTML de la página como prueba
print(driver.page_source)

# Cerrar el navegador
driver.quit()

NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# Ruta correcta al archivo chromedriver
driver_path = r'C:\Program Files\Google\Chrome\Application\chromedriver'

# Configurar las opciones de Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")  # Opcional: Ejecutar sin interfaz gráfica

# Inicializar el servicio de ChromeDriver
service = Service(driver_path)

# Inicializar el controlador de Chrome
driver = webdriver.Chrome(service=service, options=chrome_options)

# Acceder a la página
url = "https://www.example.com"
driver.get(url)

# Imprimir el código HTML como prueba
print(driver.page_source)

# Cerrar el navegador
driver.quit()


NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location


In [ ]:
pip install scraping

In [ ]:
pip install requests
pip install beautifulsoup4

SyntaxError: invalid syntax (<ipython-input-53-8b5c191d12d8>, line 1)

In [ ]:
import requests
from bs4 import BeautifulSoup

# URL de la página de MercadoLibre que vamos a scrapear
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"

# Hacer la solicitud HTTP a la página web
response = requests.get(url)

# Verificar que la solicitud fue exitosa
if response.status_code == 200:
    # Parsear el contenido HTML con BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Encontrar todos los inmuebles listados
    houses = soup.find_all("li", class_="ui-search-layout__item")

    # Iterar sobre cada inmueble y extraer información relevante
    for house in houses:
        # Obtener el título del inmueble
        title = house.find("h2", class_="ui-search-item__title")
        title_text = title.text.strip() if title else "N/A"

        # Obtener el precio del inmueble
        price = house.find("span", class_="price-tag-fraction")
        price_text = price.text.strip() if price else "N/A"

        # Obtener la dirección del inmueble
        address = house.find("span", class_="ui-search-item__location")
        address_text = address.text.strip() if address else "N/A"

        # Obtener la URL del inmueble
        url_house = house.find("a", class_="ui-search-link")['href']

        # Imprimir la información extraída
        print(f"Title: {title_text}")
        print(f"Price: {price_text}")
        print(f"Address: {address_text}")
        print(f"URL: {url_house}")
        print("="*50)

else:
    print(f"Error al acceder a la página, código de estado: {response.status_code}")

Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D421a3281-6cc0-49ea-a0cc-465f39560856
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D421a3281-6cc0-49ea-a0cc-465f39560856
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303122320-departamento-en-pre-venta-en-residencial-privado-con-club-de-playa-privado-y-parque-acuatico-y-deportivo-_JM#position%3D3%26search_layout%3Dgr

In [ ]:
import requests
from bs4 import BeautifulSoup

# URL de la página de MercadoLibre
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"

# Hacer la solicitud HTTP a la página web
response = requests.get(url)

# Verificar que la solicitud fue exitosa
if response.status_code == 200:
    # Parsear el contenido HTML con BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Encontrar todos los inmuebles listados
    houses = soup.find_all("li", class_="ui-search-layout__item")

    # Iterar sobre cada inmueble y extraer información relevante
    for house in houses:
        # Obtener el título del inmueble
        title = house.find("h2", class_="ui-search-item__title")
        title_text = title.text.strip() if title else "N/A"

        # Obtener el precio del inmueble
        price = house.find("span", class_="price-tag-fraction")
        price_text = price.text.strip() if price else "N/A"

        # Obtener la dirección del inmueble (verifica si la clase es correcta)
        address = house.find("span", class_="ui-search-item__location")
        address_text = address.text.strip() if address else "N/A"

        # Obtener la URL del inmueble
        url_house = house.find("a", class_="ui-search-link")['href']

        # Imprimir la información extraída
        print(f"Title: {title_text}")
        print(f"Price: {price_text}")
        print(f"Address: {address_text}")
        print(f"URL: {url_house}")
        print("="*50)

else:
    print(f"Error al acceder a la página, código de estado: {response.status_code}")


Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D44d706f3-7b8a-47ea-bc44-a801676ada18
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D44d706f3-7b8a-47ea-bc44-a801676ada18
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303122320-departamento-en-pre-venta-en-residencial-privado-con-club-de-playa-privado-y-parque-acuatico-y-deportivo-_JM#position%3D3%26search_layout%3Dgr

In [ ]:
import requests
from bs4 import BeautifulSoup

# URL de la página de MercadoLibre
url = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/guerrero/acapulco/"

# Hacer la solicitud HTTP a la página web
response = requests.get(url)

# Verificar que la solicitud fue exitosa
if response.status_code == 200:
    # Parsear el contenido HTML con BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Encontrar todos los inmuebles listados
    houses = soup.find_all("li", class_="ui-search-layout__item")

    # Iterar sobre cada inmueble y extraer información relevante
    for house in houses:
        # Obtener el título del inmueble
        title = house.find("h2", class_="ui-search-item__title")
        title_text = title.text.strip() if title else "N/A"

        # Obtener el precio del inmueble (Verificar si esta clase ha cambiado)
        price = house.find("span", class_="price-tag-text-sr-only")
        if price is None:
            price = house.find("span", class_="price-tag-fraction")  # Probar con otro posible selector
        price_text = price.text.strip() if price else "N/A"

        # Obtener la dirección del inmueble (Verifica la clase correcta para la ubicación)
        address = house.find("span", class_="ui-search-item__location")
        address_text = address.text.strip() if address else "N/A"

        # Obtener la URL del inmueble
        url_house = house.find("a", class_="ui-search-link")['href']

        # Imprimir la información extraída
        print(f"Title: {title_text}")
        print(f"Price: {price_text}")
        print(f"Address: {address_text}")
        print(f"URL: {url_house}")
        print("="*50)

else:
    print(f"Error al acceder a la página, código de estado: {response.status_code}")

Title: Departamento En Venta En Lomas De Costa Azul Acapulco
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-2135986661-departamento-en-venta-en-lomas-de-costa-azul-acapulco-_JM#position%3D1%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D6f94a214-8e58-4ddd-8d3e-106d00b0be38
Title: Departamento En Pre Venta A 5 Minutos De La Zona De Playas Acapulco Diamante
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303109512-departamento-en-pre-venta-a-5-minutos-de-la-zona-de-playas-acapulco-diamante-_JM#position%3D2%26search_layout%3Dgrid%26type%3Ditem%26tracking_id%3D6f94a214-8e58-4ddd-8d3e-106d00b0be38
Title: Departamento  En Pre Venta En Residencial Privado Con Club De Playa Privado Y Parque Acuático Y Deportivo
Price: N/A
Address: N/A
URL: https://departamento.mercadolibre.com.mx/MLM-3303122320-departamento-en-pre-venta-en-residencial-privado-con-club-de-playa-privado-y-parque-acuatico-y-deportivo-_JM#position%3D3%26search_layout%3Dgr

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Lista de URLs de las propiedades que deseas procesar
property_urls = [
    "https://departamento.mercadolibre.com.mx/MLM-3113424926-departamento-en-venta-en-kabah-acapulco-playa-diamante_JM",
    # Agrega las demás URLs aquí...
]

# Lista para almacenar los resultados
data = []

# Función para extraer el link de Google Maps de una propiedad
def extract_google_maps_url(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Encuentra todos los enlaces
    links = soup.find_all('a', href=True)

    # Busca el link de Google Maps
    for link in links:
        href = link['href']
        if 'google.com/maps' in href:
            return href
    return None

# Iterar sobre las URLs de las propiedades
for property_url in property_urls:
    google_maps_url = extract_google_maps_url(property_url)
    data.append({'Property URL': property_url, 'Google Maps URL': google_maps_url})

# Crear un DataFrame
df = pd.DataFrame(data)

# Guardar los resultados en un archivo CSV
df.to_csv('property_google_maps_links.csv', index=False)

print("Extracción completada y archivo CSV generado.")

Extracción completada y archivo CSV generado.


In [ ]:
import requests
from bs4 import BeautifulSoup

# Lista de URLs de las propiedades que deseas procesar
property_urls = [
    "https://departamento.mercadolibre.com.mx/MLM-3113424926-departamento-en-venta-en-kabah-acapulco-playa-diamante_JM",
    # Agrega las demás URLs aquí...
]

# Función para extraer el link de Google Maps de una propiedad
def extract_google_maps_url(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Encuentra todos los enlaces
    links = soup.find_all('a', href=True)

    # Busca el link de Google Maps
    for link in links:
        href = link['href']
        if 'google.com/maps' in href:
            return href
    return None

# Iterar sobre las URLs de las propiedades
for property_url in property_urls:
    google_maps_url = extract_google_maps_url(property_url)
    if google_maps_url:
        print(f"Property URL: {property_url}")
        print(f"Google Maps URL: {google_maps_url}\n")
    else:
        print(f"Property URL: {property_url}")
        print("Google Maps URL: No encontrado\n")

Property URL: https://departamento.mercadolibre.com.mx/MLM-3113424926-departamento-en-venta-en-kabah-acapulco-playa-diamante_JM
Google Maps URL: No encontrado



In [ ]:
import pandas as pd
import re

# Cargar el archivo CSV
df = pd.read_csv('Final_Transposed_Data.csv')

# Función para extraer las coordenadas de una URL de Google Maps
def extract_coordinates(url):
    # Utilizar expresiones regulares para encontrar las coordenadas en el formato de decimales
    match = re.search(r'@(-?\d+\.\d+),(-?\d+\.\d+)', url)
    if match:
        return f"{match.group(1)},{match.group(2)}"
    else:
        return None

# Aplicar la función a la columna que contiene las URLs de Google Maps y crear una nueva columna con las coordenadas
df['Coordenadas'] = df['Google Maps URL'].apply(extract_coordinates)

# Mostrar el resultado con la nueva columna de coordenadas
print(df[['ID', 'Google Maps URL', 'Coordenadas']])

FileNotFoundError: [Errno 2] No such file or directory: 'Final_Transposed_Data.csv'

In [ ]:
import pandas as pd
import re

# Supongamos que 'df' ya está cargado en memoria.
# Por ejemplo, si ya tienes un DataFrame llamado 'df', puedes seguir con las siguientes operaciones.

# Función para extraer las coordenadas de una URL de Google Maps
def extract_coordinates(url):
    # Utilizar expresiones regulares para encontrar las coordenadas en el formato de decimales
    match = re.search(r'@(-?\d+\.\d+),(-?\d+\.\d+)', url)
    if match:
        return f"{match.group(1)},{match.group(2)}"
    else:
        return None

# Aplicar la función a la columna que contiene las URLs de Google Maps y crear una nueva columna con las coordenadas
df['Coordenadas'] = df['Google Maps URL'].apply(extract_coordinates)

# Mostrar el resultado con la nueva columna de coordenadas
print(df[['ID', 'Google Maps URL', 'Coordenadas']])

# Devolver el DataFrame con la nueva columna de coordenadas
df

TypeError: expected string or bytes-like object

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

#---------------------------------------------------------------
# Función para insertar un inmueble y mostrar los datos en la consola
def insert_house(house_data):
    print(f"Title: {house_data['title']}")
    print(f"Price: {house_data['price_house']}")
    print(f"Size: {house_data['size']}")
    print(f"Rooms: {house_data['rooms']}")
    print(f"Location: {house_data['location']}")
    print(f"URL: {house_data['url']}")
    print("="*50)

#---------------------------------------------------------------
# Función para extraer datos de cada inmueble
def obj_house(house_html):
    # Extraer el título
    title = house_html.find("a", class_="go-to-posting")
    title = title.text.strip() if title else "N/A"

    # Extraer el precio
    price_house = house_html.find("div", class_="first-price")
    price_house = price_house.text.strip() if price_house else "N/A"

    # Extraer la ubicación
    location = house_html.find("span", class_="posting-location")
    location = location.text.strip() if location else "N/A"

    # Extraer tamaño y habitaciones
    size = "N/A"
    rooms = "N/A"

    features = house_html.find_all("li", class_="icon-feature")
    for feature in features:
        if "m²" in feature.text:
            size = feature.text.strip()
        elif "recámara" in feature.text:
            rooms = feature.text.strip()

    # Extraer la URL del inmueble
    inmueble_url = house_html.find("a", class_="go-to-posting")["href"]
    inmueble_url = "https://www.inmuebles24.com" + inmueble_url

    return {
        "title": title,
        "price_house": price_house,
        "location": location,
        "size": size,
        "rooms": rooms,
        "url": inmueble_url
    }

#---------------------------------------------------------------
# Petición GET a la URL
url = "https://www.inmuebles24.com/departamentos-en-renta-en-acapulco-de-juarez.html"
r = requests.get(url)
data = r.text

#---------------------------------------------------------------
# Convertir el HTML en un objeto BeautifulSoup
soup = BeautifulSoup(data, 'lxml')

#---------------------------------------------------------------
# Identificar los elementos que contienen los inmuebles
houses = soup.find_all("div", class_="posting-card")

#---------------------------------------------------------------
# Iterar sobre cada inmueble y extraer los datos
if houses:
    for house_html in houses:
        house_obj = obj_house(house_html)
        insert_house(house_obj)
else:
    print("No se encontraron inmuebles.")

print("Scraping completado.")


No se encontraron inmuebles.
Scraping completado.
